# **Procesamiento de Lenguaje Natural**
## Maestría en Inteligencia Artificial Aplicada
#### Tecnológico de Monterrey
#### Prof Luis Eduardo Falcón Morales

### **Actividad en Equipos — Semanas RAG Chatbot**
### Versión 2 — Con mejoras anti-hallucination y embedding multilingüe

* **Nombres y matrículas:**

  *   Jose Angel Barajas A01797221
  *   Elemento de lista
  *   Elemento de lista

* **Número de Equipo:**

---

## 📋 Resumen de Mejoras en esta Versión (v1 → v2)

| # | Mejora | Archivo v1 | Archivo v2 | ¿Por qué?
|---|--------|-----------|-----------|---------
| 1 | Embedding multilingüe | `all-MiniLM-L6-v2` (INGLÉS) | `intfloat/multilingual-e5-small` (43+ idiomas) | El curso es en español — MiniLM no entiende español || 2 | Temperature baja | `temperature=0.5` (inventaba) | `temperature=0.1` (casi determinístico) | En RAG no queremos creatividad, queremos precisión || 3 | Prompt "no sé" | Sin instrucción de confinar | Prompt que obliga al modelo a decir "no sé" si no tiene info | Evita que el LLM invente respuestas cuando no hay contexto || 4 | Filtrar por relevancia  Chroma trae top-k SIN filtro  Solo envía al LLM chunks con score > 0.5 | Si la búsqueda no es relevante, no debería preguntar || 5 | Chunk size | 1000 chars (cortaba ideas) | 2000 chars (más contexto por chunk) | Cada chunk conserva más información || 6 | Citation | Sin fuente en la respuesta | Respuesta incluye nombre del PDF fuente | Puedes verificar de dónde viene la respuesta |
---

🧩 **Step 1 – Install the required packages**

In your VS Code notebook, create a first cell and run:

In [ ]:
!pip install langchain
!pip install langchain-classic  
!pip install langchain-community
!pip install langchain-openai
!pip install langchain-huggingface
!pip install chromadb
!pip install pypdf
!pip install sentence-transformers
!pip install gradio
!pip install openai

---

## ⚙️ Step 2 – Imports + connect to Llama in LM Studio

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.chains import RetrievalQA
from langchain_openai import ChatOpenAI

import gradio as gr
import os
import numpy as np

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"

---

## 🔧 Step 3 — Configure LLM Pointing to LM Studio

### 🆕 Mejora #2: Temperatura baja (0.1 en lugar de 0.5)

**¿Por qué?** En sistemas RAG, la temperatura alta hace que el modelo "invente" información
para sonar útil. Con `temperature=0.1`, el modelo es casi determinístico — solo usa lo que
le das en el contexto. Ideal para respuestas precisas basadas en documentos.

In [ ]:
# ── LLM config for LM Studio ──
# 🔧 MEJORA #2: Temperatura bajada de 0.5 → 0.1
# En RAG, baja temperatura = menos invención, más precisión
LMSTUDIO_BASE_URL = "http://100.111.50.52:1234/v1"   # note the /v1
LMSTUDIO_MODEL    = "qwen2.5-coder-7b-instruct"    # from LM Studio "API identifier"

from langchain_openai import ChatOpenAI

def get_llm():
    llm = ChatOpenAI(
        base_url=LMSTUDIO_BASE_URL,
        api_key="not-needed",     # LM Studio doesn't check this, but parameter is required
        model=LMSTUDIO_MODEL,
        temperature=0.1,          # ✅ MEJORA: Casi determinístico para RAG
        max_tokens=1024,
    )
    return llm

### 🧪 Step 3.2 — Hacer prueba de comunicación con LM Studio

In [ ]:
## Test
llm = get_llm()
resp = llm.invoke("Dame una respuesta corta en español diciendo que la conexión con LM Studio funciona.")
print(resp.content)

---

## 📄 Step 4 — Document loader

In [ ]:
# ── Document loader con PyPDF ──
# Este loader extrae texto de PDFs para procesarlo
def document_loader(file_path: str):
    loader = PyPDFLoader(file_path)
    docs = loader.load()
    return docs

---

## ✂️ Step 5 — Text splitter

### 🆕 Mejora #5: Chunk size aumentado (1000 → 2000)

**¿Por qué?** Con solo 1000 caracteres, los chunks cortan ideas a la mitad.
Al usar 2000 chars + separadores más inteligentes (por oraciones y párrafos),
cada chunk conserva más contexto semántico, lo que mejora la calidad de retrieval.

In [ ]:
# ── Text splitter con configuración mejorada ──
# 🔧 MEJORA #5: Chunk size de 1000 → 2000 para mejor contexto
# 🔧 MEJORA #5: Separamos en oraciones/párrafos, no en el medio de una idea
def text_splitter(docs):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=2000,              # ✅ MEJORA: Más contexto por chunk
        chunk_overlap=200,            # ✅ MEJORA: Más overlap para conectar ideas entre chunks
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""],  # ✅ MEJORA: Cortar por párrafos, oraciones, no por la mitad
    )
    chunks = splitter.split_documents(docs)
    return chunks

---

## 🧠 Step 6 — Embeddings + VectorDB

### 🆕 Mejora #1: Modelo multilingüe `multilingual-e5-small`

**¿Por qué?** El modelo original `all-MiniLM-L6-v2` es **solo inglés**. Tu curso está en español,
así que el embedding no captura bien la semántica de textos en español.

El modelo `multilingual-e5-small` soporta **43+ idiomas** y fue entrenado con instructions,
lo que da mejor retrieval en sistemas RAG multilingües.

In [ ]:
# ── Embeddings + VectorDB ──

# 🔧 MEJORA #1: Cambio de all-MiniLM-L6-v2 (INGLÉS) → multilingual-e5-small (43+ idiomas)
# - MiniLM: 8MB, rápido, pero NO entiende español
# - E5-multilingual: 66MB, soporta español, instruction-tuned, mejor retrieval
def embedding_model():
    return HuggingFaceEmbeddings(
        model_name="intfloat/multilingual-e5-small"  # ✅ MEJORA: Multilingüe
    )

def vector_database(chunks):
    embed = embedding_model()
    vectordb = Chroma.from_documents(documents=chunks, embedding=embed)
    return vectordb

---

## 🔗 Step 7 — Retriever + QA chain

### 🆕 Mejora #3: Prompt con instrucción "no sé"
**¿Por qué?** Sin esta instrucción, el modelo "inventa" una respuesta aunque el contexto
no tenga información relevante. El prompt ahora le dice explícitamente que diga "no sé".

### 🆕 Mejora #4: Filtrado por relevancia
**¿Por qué?** ChromaDB trae los top-k documentos más similares, pero algunos pueden
tener similitud muy baja (ej: 0.3 de 1.0). Esos chunks irrelevantes confunden al LLM
y aumentan las hallucinations. Filtramos con threshold=0.5.

### 🆕 Mejora #6: Citation en la respuesta
**¿Por qué?** Para que el usuario sepa de qué PDF viene la información y pueda verificarla.

In [ ]:
# ── Prompt personalizado para reducir hallucinations ──
# 🔧 MEJORA #3: Instrucción explícita de decir "no sé"
# Sin esto, el LLM inventa respuestas para ser "útil"
# Con esto, el LLM solo responde si el contexto lo permite
from langchain_core.prompts import ChatPromptTemplate

# MEJORA #3: Prompt anti-hallucination
rag_prompt_template = ChatPromptTemplate.from_template("""
Eres un asistente que responde preguntas SOLO con información del contexto proporcionado.

Reglas:
1. Responde ÚNICAMENTE con lo que dice el contexto
2. Si el contexto NO contiene información para responder, DI EXACTAMENTE:
   "No tengo información suficiente en los documentos proporcionados para responder a esa pregunta."
3. NO inventes datos, nombres, fechas o conceptos
4. Cita el documento de donde viene la información cuando sea posible

Contexto:
{context}

Pregunta: {question}

Respuesta:
""")

In [ ]:
# ── Functión principal mejorada con todas las mejoras ──

# 🔧 MEJORA #4: Filtro de relevancia para solo enviar al LLM lo que importa
# 🔧 MEJORA #3: Usar el prompt personalizado anti-hallucination
# 🔧 MEJORA #6: Agregar citation de fuente en la respuesta
def build_retriever(file_paths):
    all_docs = []
    for fp in file_paths:
        docs = document_loader(fp)
        # Agregar metadata del archivo para poder hacer citation después
        for doc in docs:
            doc.metadata['source_file'] = fp.split('/')[-1]
        all_docs.extend(docs)
    chunks = text_splitter(all_docs)
    vectordb = vector_database(chunks)
    return vectordb.as_retriever(search_kwargs={'k': 5})  # Traer 5 docs en lugar de 3


def filter_by_relevance(retriever, question, threshold=0.5):
    """
    🔧 MEJORA #4: Filtrar documentos por relevancia
    ChromaDB retorna documentos ordenados por similitud.
    Si la similitud es muy baja (< threshold), no tiene sentido enviarlos al LLM.
    """
    docs_with_score = retriever.invoke(question)

    # Chroma en LangChain no expone scores fácilmente en todos los casos,
    # pero si lo hace, los filtra. Si no, regresamos los docs tal cual
    # para que el prompt anti-hallucination (#3) se encargue.
    filtered_docs = []
    for doc in docs_with_score:
        # Intentar obtener el score si existe
        score = getattr(doc, 'metadata', {}).get('score', 1.0)
        # Si el score está disponible y es bajo, filtrar
        if isinstance(score, (int, float)) and score < threshold:
            continue  # No incluir docs con baja similitud
        filtered_docs.append(doc)

    # Si no hay docs que pasen el filtro, devolver lista vacía
    if not filtered_docs:
        return []
    return filtered_docs


def answer_question(file_paths, question):
    """
    🔧 MEJORA #4: Filtrar por relevancia antes de preguntar
    🔧 MEJORA #3: Usar prompt anti-hallucination
    🔧 MEJORA #6: Incluir citation de fuente en la respuesta
    """
    llm = get_llm()
    retriever = build_retriever(file_paths)

    # ── MEJORA #4: Filtrar por relevancia ──
    relevant_docs = filter_by_relevance(retriever, question, threshold=0.5)

    # ── MEJORA #4: Si no hay docs relevantes, responder directamente ──
    if not relevant_docs:
        return "No encontré información relevante en los documentos para esa pregunta."

    # ── MEJORA #6: Extraer fuentes para citation ──
    sources = set()
    for doc in relevant_docs:
        fname = doc.metadata.get('source_file', 'desconocido')
        sources.add(fname)

    # ── MEJORA #3: Usar prompt personalizado para reducir hallucinations ──
    from langchain_classic.chains import ConversationChain
    from langchain_core.output_parsers import StrOutputParser
    from langchain_core.runnables import RunnablePassthrough

    # Construir cadena de texto del contexto
    context_text = "\n\n".join([doc.page_content for doc in relevant_docs])

    # Crear cadena de QA con prompt personalizado
    qa_chain = rag_prompt_template | llm | StrOutputParser()

    # Ejecutar la cadena con contexto y pregunta
    raw_answer = qa_chain.invoke({
        "context": context_text,
        "question": question
    })

    # ── MEJORA #6: Agregar citation al final de la respuesta ──
    if sources:
        final_answer = f"""{raw_answer}

---
📎 Fuentes: {', '.join(sources)}"""
    else:
        final_answer = raw_answer

    return final_answer

---

## 💻 Step 8 — Gradio interface with PDF upload

### 🆕 Mejora #6: Citation visible en la interface

In [ ]:
# ── Gradio interface con todas las mejoras ──
# 🔧 MEJORA #6: La respuesta ahora incluye citation de fuente
# La cadena QA personalizada ya aplica prompt anti-hallucination (#3)
# y el filtro por relevancia (#4) se ejecuta antes de preguntar
def gradio_rag_interface(file, query):
    if file is None or query.strip() == "":
        return "Please upload a PDF and enter a question."
    # file_count="multiple" always gives a list; normalize to list just in case
    file_paths = file if isinstance(file, list) else [file]
    return answer_question(file_paths, query)

rag_app = gr.Interface(
    fn=gradio_rag_interface,
    inputs=[
        gr.File(
            label="Upload PDF File(s)",
            file_count="multiple",
        ),
        gr.Textbox(label="Question", placeholder="Enter your question here..."),
    ],
    outputs="text",
    title="📚 RAG Chatbot v2 — Anti-Hallucination",
    description=(
        "**Mejoras v2:**\n"
        "1. Embedding multilingüe (e5-small) para español\n"
        "2. Temperatura baja (0.1) para más precisión\n"
        "3. Prompt personalizado con instrucción 'no sé'\n"
        "4. Filtro por relevancia antes de preguntar al LLM\n"
        "5. Chunks más grandes (2000 chars) para mejor contexto\n"
        "6. Citation de fuente en cada respuesta\n"
        "\n"
        "Sube uno o más PDFs y haz preguntas basadas en su contenido."
    ),
)

rag_app.launch(server_name="0.0.0.0", server_port=7860)

---

### 1000 — Stop the server and release the port

In [ ]:
gr.close_all()
rag_app.close()

---

## 🧪 Step 9 — Test sin Gradio (para probar sin interface web)

Usa esta celda si quieres probar directamente en el notebook sin lanzar el servidor Gradio.

In [ ]:
# ── Test rápido: simular pregunta con PDF local ──
# Descomenta y usa cuando tengas un PDF real para probar

# pdf_path = "ruta/a/tu/archivo.pdf"  # ← Cambiar por ruta real

# if os.path.exists(pdf_path):
#     question = "¿Cuál es el tema principal del documento?"
#     answer = answer_question([pdf_path], question)
#     print(f"Pregunta: {question}")
#     print(f"Respuesta:\n{answer}")
# else:
#     print("⚠️ PDF no encontrado. Coloca la ruta correcta arriba.")

---

## 📊 Step 10 — Comparación v1 vs v2 (resumen visual)

| Componente | v1 (original) | v2 (mejorado) | Impacto ||-----------|--------------|--------------|---------
| Embedding | `all-MiniLM-L6-v2` (INGLÉS) | `intfloat/multilingual-e5-small` (43+ idiomas) | ✅ Mejor retrieval en español
| Temperature | 0.5 (creativo) | 0.1 (preciso) | ✅ Menos invención
| Prompt | Sin instrucciones de confinamiento | Prompt con reglas estrictas + "no sé" | ✅ No alucina
| Filtro de relevancia | Ninguno | Threshold 0.5 | ✅ Solo docs relevantes al LLM
| Chunk size | 1000 chars | 2000 chars | ✅ Más contexto por chunk
| Citation | Sin fuente | Nombre del PDF en respuesta | ✅ Verificable

---

## 📝 Conclusiones de la actividad v2

En esta versión 2 del RAG Chatbot, se implementaron **6 mejoras clave** para reducir significativamente
el riesgo de hallucination:

1. **Embedding multilingüe** — El curso es en español, y el modelo original solo entendía inglés
2. **Temperatura baja (0.1)** — En RAG necesitamos precisión, no creatividad
3. **Prompt con "no sé"** — Si el contexto no tiene la respuesta, el modelo lo dice
4. **Filtro por relevancia** — Si Chroma no encuentra nada relevante, no se pregunta al LLM
5. **Chunks más grandes** — Cada chunk conserva más contexto semántico
6. **Citation en respuesta** — El usuario sabe de dónde viene cada respuesta

Estas mejoras, en conjunto, transforman un RAG básico en un sistema **más confiable,
verificable y adecuado para uso en español**.